# Этап 1: smoke-тест инфраструктуры

Проверяем, что локальный Python и Spark видят все сервисы из Docker:

1. **Kafka** - отправляем 5 событий в тестовый топик `smoke_test`.
2. **RustFS (S3)** - создаем бакет `raw`.
3. **Spark** - читаем события из Kafka и записываем их в S3 в формате Parquet.
4. **ClickHouse** - выполняем простой запрос.

Запускай ячейки по очереди  и сверяйся с ожидаемым результатом под каждой.

In [ ]:
# --- Настройки ---
# Загружаем переменные из файла .env в корне проекта.
# Ноутбук лежит в notebooks/, поэтому .env на уровень выше.
import os
import sys 
import json
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

# Адреса берем из окружения; значения после запятой запасные, для локального запуска.
KAFKA_BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP", "localhost:9094")
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "http://localhost:9000")
CH_HOST = os.getenv("CH_HOST", "localhost")
CH_PORT = int(os.getenv("CH_PORT", "8123"))

# Пароли только из .env, без запасных значений: если их нет, лучше сразу упасть.
S3_ACCESS_KEY = os.environ["S3_ACCESS_KEY"]
S3_SECRET_KEY = os.environ["S3_SECRET_KEY"]
CH_USER = os.environ["CLICKHOUSE_USER"]
CH_PASSWORD = os.environ["CLICKHOUSE_PASSWORD"]

# Даем ноутбуку найти наш модуль jobs/events.py
sys.path.append(str(PROJECT_ROOT / "jobs"))
from events import generate_event

print("Kafka:", KAFKA_BOOTSTRAP, "| S3:", S3_ENDPOINT, "| ClickHouse:", f"{CH_HOST}:{CH_PORT}")

Ожидаем строку с тремя адресами. Ошибка `KeyError: 'S3_ACCESS_KEY'` значит, что не найден файл `.env` в корне проекта.

In [ ]:
# --- 1. Kafka: отправляем 5 событий ---
from confluent_kafka import Producer

producer = Producer({"bootstrap.servers": KAFKA_BOOTSTRAP})

for _ in range(5):
    event = generate_event()
    # Kafka хранит байты: dict -> JSON-строка -> байты
    producer.produce("smoke_test", value=json.dumps(event).encode("utf-8"))

# produce только ставит сообщения в очередь; flush ждет, пока они реально уйдут в Kafka.
# Возвращает число НЕ доставленных сообщений.
not_delivered = producer.flush(15)
print("Не доставлено:", not_delivered)

Ожидаем `Не доставлено: 0`. Топик `smoke_test` создастся автоматически при первой отправке.

In [ ]:
# --- 2. S3 (RustFS): создаем бакет raw ---
# boto3 — официальная библиотека AWS. RustFS говорит на том же протоколе S3,
# поэтому достаточно указать его адрес в endpoint_url.
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    region_name="us-east-1",   # RustFS регион не важен, но boto3 требует его указать
)

existing = [b["Name"] for b in s3.list_buckets()["Buckets"]]
if "raw" not in existing:
    s3.create_bucket(Bucket="raw")

print("Бакеты:", [b["Name"] for b in s3.list_buckets()["Buckets"]])

Ожидаем `Бакеты: ['raw']`. Бакет будет виден и в веб-консоли http://localhost:9001.

In [ ]:
# --- 3. Spark: запуск сессии ---
# При ПЕРВОМ запуске Spark скачает jar-пакеты из Maven (~300 МБ) в ~/.ivy2 —
# это займет несколько минут. Дальше они берутся из кэша.
import pyspark
from pyspark.sql import SparkSession

SPARK_VERSION = pyspark.__version__   # версия коннектора Kafka должна совпадать с версией Spark

packages = ",".join([
    f"org.apache.spark:spark-sql-kafka-0-10_2.12:{SPARK_VERSION}",  # чтение из Kafka
    "org.apache.hadoop:hadoop-aws:3.3.4",                             # протокол s3a://
    "com.amazonaws:aws-java-sdk-bundle:1.12.262",                     # AWS SDK для hadoop-aws
])

spark = (
    SparkSession.builder
    .appName("smoke-test")
    .master("local[*]")                       # local-режим: все ядра этого компьютера
    .config("spark.jars.packages", packages)
    .config("spark.driver.memory", "2g")
    # Настройки доступа к S3 (RustFS) через s3a://
    .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", S3_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", S3_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")       # адрес вида host/бакет, а не бакет.host
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") # локально без HTTPS
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

hadoop_version = spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
print("Spark:", spark.version, "| Hadoop:", hadoop_version)

Ожидаем `Spark: 3.5.x | Hadoop: 3.3.4`. Если Hadoop другой версию `hadoop-aws` нужно будет поменять под него.

In [ ]:
# --- 4. Spark читает события из Kafka ---
# spark.read обычное разовое чтение всего, что есть в топике.
# Стриминг будет на следующем этапе
raw = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", "smoke_test")
    .option("startingOffsets", "earliest")   # с самого начала топика
    .load()
)

# Kafka отдает value в байтах,  приводим к строке. partition и offset "координаты" сообщения.
events = raw.selectExpr("CAST(value AS STRING) AS json", "partition", "offset")
events.show(truncate=False)

Ожидаем таблицу из 5 строк с JSON событий (или больше, если ячейку с отправкой запускали несколько раз).

In [ ]:
# --- 5. Spark пишет в S3 и читает обратно ---
path = "s3a://raw/smoke_test/"
events.write.mode("overwrite").parquet(path)

back = spark.read.parquet(path)
print("Прочитано из S3 строк:", back.count())

Ожидаем то же число строк, что в таблице выше. В консоли RustFS (http://localhost:9001) в бакете `raw` появится папка `smoke_test` с parquet-файлами.

In [ ]:
# --- 6. ClickHouse: простой запрос ---
import clickhouse_connect

ch = clickhouse_connect.get_client(
    host=CH_HOST, port=CH_PORT, username=CH_USER, password=CH_PASSWORD
)
print("ClickHouse версии", ch.query("SELECT version()").result_rows[0][0])

Ожидаем `ClickHouse версии 24.8...`. Если все шесть шагов прошли то инфраструктура готова.

In [ ]:
# Останавливаем Spark, чтобы он освободил память.
spark.stop()